# Captain Pairing — Eurowings OR Practice Project
## Week 1 · DUS Base · Column Generation + RCSP Pricing

---

### How this notebook is structured

| Phase | What we do | Algorithm |
|---|---|---|
| **1 · Data** | Load & inspect flights, EASA limits, ground times | — |
| **2 · Network** | Model the week as a **Time-Space Network** (nodes = flight events, arcs = feasible connections) | Directed graph |
| **3 · Duty generation** | Enumerate legal single-day duties via **Resource-Constrained Shortest Path (RCSP)** on the daily sub-graph | DFS + resource pruning |
| **4 · Set Partitioning** | Formulate the Restricted Master Problem (RMP): every leg covered **exactly once** | Integer Program |
| **5 · Column Generation** | Solve LP relaxation → extract dual prices → solve **Pricing Sub-Problem** → add negative-reduced-cost columns | CG loop |
| **6 · Integer solve** | Fix integrality (branch-and-bound via Gurobi) | MILP |
| **7 · Validation** | Verify every leg is covered, every pairing is legal | Checks |

### Key OR concepts used here

**Set Partitioning Problem (SPP)**  
The master problem: choose a subset of pre-generated pairings so that each flight leg appears in *exactly one* chosen pairing, at minimum total cost.

**Set Covering Problem (SCP)**  
A relaxed version (≥ 1 instead of = 1). Useful as a warm-start, but SPP is what the airline actually needs.

**Column Generation (CG)**  
Because the full set of legal pairings is astronomically large (exponential in flight count), we never enumerate them all. Instead we start with a small *Restricted Master Problem* (RMP), solve it to get dual variables (shadow prices), then solve a *Pricing Sub-Problem* to find the single most profitable column to add. We repeat until no improving column exists → LP optimum reached.

**Pricing Sub-Problem / RCSP**  
Given dual prices on each leg, a new pairing has *negative reduced cost* if its savings (sum of duals on its legs) exceed its raw cost. Finding the cheapest path through the time-space network subject to EASA resource constraints (FDT, rest, return-to-base) is exactly a **Resource-Constrained Shortest Path Problem (RCSP)**. We solve it with label-setting DFS.

**Time-Space Network**  
Nodes represent (airport, time) events — specifically each flight's departure and arrival event. Arcs represent either flying a leg, a legal connection between legs, or a ground transfer. A pairing is a *path* through this network from the home base at time 0 back to the home base after ≤ 7 days.


## 1 · Setup & Imports

In [ ]:
# Install if needed — comment out if already installed
import sys
# !{sys.executable} -m pip install pandas numpy networkx pulp gurobipy --quiet

import pandas as pd
import numpy as np
import networkx as nx
from collections import defaultdict
import itertools
import warnings
warnings.filterwarnings('ignore')

# --- Try Gurobi; fall back to PuLP/CBC so the notebook still runs without a licence ---
try:
    import gurobipy as gp
    from gurobipy import GRB
    SOLVER = 'gurobi'
    print("✅ Gurobi found — will use Gurobi for LP and MILP solves.")
except ImportError:
    import pulp
    SOLVER = 'pulp'
    print("⚠️  Gurobi not found — falling back to PuLP/CBC (slower, but correct).")

print(f"Pandas {pd.__version__} | NumPy {np.__version__} | NetworkX {nx.__version__}")


## 2 · Operational Constants

In [ ]:
# ── Time constants (all in MINUTES) ────────────────────────────────────────
CHECKIN_FLIGHT   = 70   # EASA: 70 min before first flight
CHECKIN_DEADHEAD = 60   # EASA: 60 min if first activity is deadhead/ground
CHECKOUT         = 30   # EASA: 30 min after last flight arrival
MIN_REST         = 720  # EASA: 12 h minimum rest between duties
MCT              = 45   # Minimum Connection Time between two flights at same airport

# ── Weekly resource limits (EASA) ───────────────────────────────────────────
MAX_DUTY_WEEK    = 60 * 60      # 60 h/week in minutes
MAX_FT_4WEEKS    = 100 * 60     # 100 h/4-weeks in minutes (we track per pairing)
MAX_PAIRING_DAYS = 7            # Pairings are at most 7 days

# ── Cost parameters (arbitrary monetary units) ──────────────────────────────
COST_PER_FLIGHT_MIN  = 1.0      # Direct operating cost per flying minute
COST_IDLE_PER_MIN    = 0.5      # Cost of crew sitting idle (duty time - flight time)
COST_HOTEL_PER_NIGHT = 150.0    # Outstation overnight
COST_GROUND_FIXED    = 50.0     # Fixed cost per ground transfer
COST_GROUND_PER_MIN  = 0.50     # Variable ground transfer cost per minute
PENALTY_ARTIFICIAL   = 500_000  # Penalty for an artificial (infeasible) column

# ── Scope ────────────────────────────────────────────────────────────────────
HOME_BASE    = 'DUS'    # Solve for Düsseldorf captains first
WEEK_START   = '2025-04-27'
WEEK_END     = '2025-05-03'

print("Constants loaded.")


## 3 · Data Loading & Preprocessing

In [ ]:
# ── 3.1  Flight schedule ─────────────────────────────────────────────────────
DATA_PATH = './'   # ← adjust if your CSVs are in a subfolder

flights_raw = pd.read_csv(DATA_PATH + 'flight_schedule.csv')
flights_raw['SCHEDULED_DEPARTURE_TIME'] = pd.to_datetime(flights_raw['SCHEDULED_DEPARTURE_TIME'])
flights_raw['SCHEDULED_ARRIVAL_TIME']   = pd.to_datetime(flights_raw['SCHEDULED_ARRIVAL_TIME'])

# Filter: Week 1 only
week_mask = (
    (flights_raw['SCHEDULED_DEPARTURE_TIME'] >= WEEK_START) &
    (flights_raw['SCHEDULED_DEPARTURE_TIME'] <= WEEK_END + ' 23:59:59')
)
flights_week = flights_raw[week_mask].copy().reset_index(drop=True)

# Compute absolute minute offsets from the start of Week 1 (makes arithmetic easy)
EPOCH = pd.Timestamp(WEEK_START)
flights_week['DEP_MIN'] = (flights_week['SCHEDULED_DEPARTURE_TIME'] - EPOCH).dt.total_seconds() / 60
flights_week['ARR_MIN'] = (flights_week['SCHEDULED_ARRIVAL_TIME']   - EPOCH).dt.total_seconds() / 60
flights_week['FLT_DUR'] = flights_week['ARR_MIN'] - flights_week['DEP_MIN']

print(f"Total legs in Week 1 : {len(flights_week)}")
print(f"Unique dep airports  : {flights_week['DEPARTURE_AIRPORT'].nunique()}")
print(f"Date range           : {flights_week['SCHEDULED_DEPARTURE_TIME'].min().date()} "
      f"→ {flights_week['SCHEDULED_DEPARTURE_TIME'].max().date()}")


In [ ]:
# ── 3.2  Scope: legs departing FROM home base ────────────────────────────────
#
# DESIGN DECISION (important to understand):
# In the full problem every leg must be covered by SOME captain from SOME base.
# For this single-base prototype we cover only legs that DEPART from DUS.
# A DUS captain's pairing looks like:
#   DUS → [outstation] → DUS  (round-trip, 1 duty)
#   DUS → [out1] → [out2] → DUS  (multi-day, 2+ duties with overnight at out1)
#
# The return leg (outstation → DUS) is "needed" to get the captain home,
# but it may already be covered by another DUS pairing or another base's captain.
# Here we track it only as part of the pairing structure (ensures legal return).
#
# To scale later: loop over each base and solve each sub-problem independently
# (bases are largely decoupled because captains must return home).

scope_flights = flights_week[flights_week['DEPARTURE_AIRPORT'] == HOME_BASE].copy()
scope_flights = scope_flights.sort_values('DEP_MIN').reset_index(drop=True)

# We also need all flights that could serve as return legs (arr = HOME_BASE)
return_flights = flights_week[flights_week['ARRIVAL_AIRPORT'] == HOME_BASE].copy()

# Full leg pool used during pairing construction:
# anything that departs OR arrives at DUS, or connects two such legs
# (we'll use the full week graph in the network below)
LEG_IDS_TO_COVER = set(scope_flights['LEG_ID'].tolist())

print(f"Legs departing {HOME_BASE} in Week 1: {len(scope_flights)}")
print(f"Legs arriving  {HOME_BASE} in Week 1: {len(return_flights)}")
print(f"We must cover {len(LEG_IDS_TO_COVER)} legs in the SPP.")


In [ ]:
# ── 3.3  Ground transportation times ────────────────────────────────────────
gt_raw = pd.read_csv(DATA_PATH + 'ground_transportation_times.csv')

# avg_duration is in HOURS → convert to minutes
GROUND_MAP = {
    (row['Dep Ap'].strip(), row['Arr Ap'].strip()): row['avg_duration'] * 60
    for _, row in gt_raw.iterrows()
}
print(f"Ground transfer pairs loaded: {len(GROUND_MAP)}")
print("Sample:", list(GROUND_MAP.items())[:3])


In [ ]:
# ── 3.4  EASA FDT limits ────────────────────────────────────────────────────
fdt_raw = pd.read_csv(DATA_PATH + 'fdt_limits.csv')

# Build a list of (start_hour, end_hour, {sectors: max_fdt_minutes})
EASA_WINDOWS = []
sector_cols = [c for c in fdt_raw.columns if c.isdigit()]

for _, row in fdt_raw.iterrows():
    sh = pd.to_timedelta(str(row['flight_duty_time_start_time'])).total_seconds() / 3600
    eh = pd.to_timedelta(str(row['flight_duty_time_end_time'])).total_seconds()   / 3600
    limits = {int(c): int(float(row[c]) * 60) for c in sector_cols}
    EASA_WINDOWS.append((sh, eh, limits))

def max_fdt_minutes(checkin_abs_min: float, num_sectors: int) -> int:
    """
    Return the EASA maximum FDT (in minutes) for a duty that activates at
    `checkin_abs_min` minutes from EPOCH with `num_sectors` flight legs.
    """
    hod = (checkin_abs_min % 1440) / 60.0          # hour-of-day
    for sh, eh, limits in EASA_WINDOWS:
        if sh <= eh:
            match = sh <= hod <= eh
        else:                                       # window crosses midnight
            match = hod >= sh or hod <= eh
        if match:
            s = min(num_sectors, max(limits.keys()))
            return limits[s]
    return 9 * 60                                   # safe fallback: 9 h

# Quick sanity check
print("Check-in 06:00, 2 sectors →", max_fdt_minutes(360, 2), "min")
print("Check-in 02:00, 4 sectors →", max_fdt_minutes(120, 4), "min")


## 4 · Time-Space Network Construction

### What is a Time-Space Network?

Instead of thinking about flights as isolated objects, we model the crew scheduling week as a **directed graph** where:

- Every flight leg `f` creates **two nodes**: a *departure event* `(LEG_ID, 'dep')` and an *arrival event* `(LEG_ID, 'arr')`.
- A **source node** `SRC` represents the captain signing on at DUS at the start of the week.
- A **sink node** `SNK` represents the captain signing off at DUS at the end of the week.

**Arc types:**

| Arc | Meaning | Cost |
|---|---|---|
| `dep → arr` | Flying the leg | flight time cost |
| `arr_i → dep_j` (same airport, MCT satisfied) | Direct connection | idle time cost |
| `arr_i → dep_j` (different airports, ground transfer feasible) | Ground transfer | transfer cost |
| `arr_i → arr_i` (rest arc) | Overnight rest at outstation (≥ 12 h gap) | hotel cost |
| `SRC → dep_j` | Start duty from DUS | check-in cost |
| `arr_i → SNK` | Return to DUS after last duty | check-out cost |

A **legal pairing** is any path `SRC → ... → SNK` where:
- Each arc's resource consumption keeps all running totals (FDT, flight time, rest) within EASA limits.
- The path starts and ends at DUS.
- Total duration ≤ 7 days.

This is why the professor said "convert it to a network problem" — once it's a network, the optimal solution is the **cheapest feasible path**, which connects directly to RCSP and Column Generation.


In [ ]:
# ── 4.1  Build the Time-Space Network ───────────────────────────────────────
#
# Nodes: one per flight-arrival and flight-departure event
# We limit the graph to flights reachable from DUS (depart DUS, or arrive DUS,
# or link two such flights at an outstation).

# Collect the reachable leg pool:
#   Tier 1: legs departing DUS
#   Tier 2: legs arriving DUS (return trips)
#   Tier 3: any leg that connects Tier1-arrival-airport → Tier2 or is part of
#            a multi-sector duty starting at DUS.
# For simplicity in Week 1, we use Tier 1 + Tier 2 + all legs whose
# departure airport appears in Tier-1 arrival airports.

tier1_arr_airports = set(scope_flights['ARRIVAL_AIRPORT'].unique())

tier3_mask = flights_week['DEPARTURE_AIRPORT'].isin(tier1_arr_airports)
tier3_flights = flights_week[tier3_mask & (flights_week['ARRIVAL_AIRPORT'] == HOME_BASE)]

leg_pool = pd.concat([scope_flights, tier3_flights]).drop_duplicates('LEG_ID')
leg_pool = leg_pool.sort_values('DEP_MIN').reset_index(drop=True)

print(f"Leg pool size (DUS-reachable): {len(leg_pool)}")

# Build flight lookup maps
LEG_DEP_MIN  = dict(zip(leg_pool['LEG_ID'], leg_pool['DEP_MIN']))
LEG_ARR_MIN  = dict(zip(leg_pool['LEG_ID'], leg_pool['ARR_MIN']))
LEG_DEP_AP   = dict(zip(leg_pool['LEG_ID'], leg_pool['DEPARTURE_AIRPORT']))
LEG_ARR_AP   = dict(zip(leg_pool['LEG_ID'], leg_pool['ARRIVAL_AIRPORT']))
LEG_FLT_DUR  = dict(zip(leg_pool['LEG_ID'], leg_pool['FLT_DUR']))
ALL_LEG_IDS  = leg_pool['LEG_ID'].tolist()


In [ ]:
# ── 4.2  Build NetworkX graph ────────────────────────────────────────────────
G = nx.DiGraph()

# Add flight arcs (dep_event → arr_event for each leg)
for lid in ALL_LEG_IDS:
    dep_node = (lid, 'dep')
    arr_node = (lid, 'arr')
    G.add_node(dep_node, airport=LEG_DEP_AP[lid], time=LEG_DEP_MIN[lid],  leg=lid)
    G.add_node(arr_node, airport=LEG_ARR_AP[lid], time=LEG_ARR_MIN[lid], leg=lid)
    flight_cost = LEG_FLT_DUR[lid] * COST_PER_FLIGHT_MIN
    G.add_edge(dep_node, arr_node, kind='flight', leg=lid, cost=flight_cost, time=LEG_FLT_DUR[lid])

# Add connection arcs between consecutive flights (within a single duty day)
# and overnight arcs (rest between duties)
for i, lid_a in enumerate(ALL_LEG_IDS):
    arr_ap_a = LEG_ARR_AP[lid_a]
    arr_min_a = LEG_ARR_MIN[lid_a]
    arr_node_a = (lid_a, 'arr')

    for lid_b in ALL_LEG_IDS:
        if lid_b == lid_a:
            continue
        dep_ap_b  = LEG_DEP_AP[lid_b]
        dep_min_b = LEG_DEP_MIN[lid_b]
        dep_node_b = (lid_b, 'dep')

        # Must be forward in time
        gap = dep_min_b - arr_min_a
        if gap < 0:
            continue

        # ── SAME-AIRPORT CONNECTION (within a duty) ──
        if arr_ap_a == dep_ap_b:
            if MCT <= gap <= 240:   # 4 h max intra-duty idle
                idle_cost = gap * COST_IDLE_PER_MIN
                G.add_edge(arr_node_a, dep_node_b,
                           kind='connection', cost=idle_cost,
                           gap=gap, transfer_type='direct')

        # ── GROUND TRANSFER CONNECTION (within a duty) ──
        elif (arr_ap_a, dep_ap_b) in GROUND_MAP:
            transit_mins = GROUND_MAP[(arr_ap_a, dep_ap_b)]
            required = transit_mins + MCT
            if required <= gap <= 240:
                transit_cost = COST_GROUND_FIXED + transit_mins * COST_GROUND_PER_MIN
                idle_cost    = (gap - transit_mins) * COST_IDLE_PER_MIN
                G.add_edge(arr_node_a, dep_node_b,
                           kind='ground_transfer', cost=transit_cost + idle_cost,
                           gap=gap, transfer_type='ground', transit_mins=transit_mins)

        # ── OVERNIGHT / REST ARC (between duties) ──
        elif gap >= MIN_REST and arr_ap_a == dep_ap_b:
            # Same airport, long rest → overnight at outstation
            hotel_cost = COST_HOTEL_PER_NIGHT * int(gap // 1440 + 1)
            G.add_edge(arr_node_a, dep_node_b,
                       kind='overnight', cost=hotel_cost,
                       gap=gap, transfer_type='overnight')

# Source and sink
G.add_node('SRC', airport=HOME_BASE, time=-CHECKIN_FLIGHT)
G.add_node('SNK', airport=HOME_BASE, time=7*1440)

# SRC → departure events that start at DUS
for lid in scope_flights['LEG_ID']:
    dep_node = (lid, 'dep')
    G.add_edge('SRC', dep_node, kind='start', cost=0, leg=lid)

# Arrival events at DUS → SNK
for lid in ALL_LEG_IDS:
    if LEG_ARR_AP[lid] == HOME_BASE:
        arr_node = (lid, 'arr')
        G.add_edge(arr_node, 'SNK', kind='end', cost=0)

print(f"Time-Space Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} arcs")
print(f"  Flight arcs      : {sum(1 for _,_,d in G.edges(data=True) if d['kind']=='flight')}")
print(f"  Connection arcs  : {sum(1 for _,_,d in G.edges(data=True) if d['kind']=='connection')}")
print(f"  Ground transfer  : {sum(1 for _,_,d in G.edges(data=True) if d['kind']=='ground_transfer')}")
print(f"  Overnight arcs   : {sum(1 for _,_,d in G.edges(data=True) if d['kind']=='overnight')}")


## 5 · Duty Generation via RCSP (Resource-Constrained Shortest Path)

### What is RCSP?

A **Resource-Constrained Shortest Path Problem** asks: find the minimum-cost path from source to sink in a network, subject to the condition that accumulated *resources* along the path stay within bounds.

Here, the resources are EASA constraints:
- **FDT** (Flight Duty Time): time from check-in to check-out of the last flight in a duty, bounded by the activation-hour/sector table.
- **Flight Time**: sum of actual flying minutes, bounded per duty and cumulatively.
- **Number of sectors**: used to look up the FDT limit.

We solve RCSP via **label-setting DFS** (depth-first search with resource pruning): we extend a partial path (label) only if adding the next leg keeps all resources feasible. This is correct and complete for a DAG (our time-space network is acyclic by construction because time always moves forward).

**This produces the column pool for Column Generation.**

A "duty" here means a single working day: check-in → [legs / ground transfers] → check-out.  
A "pairing" chains multiple duties separated by overnight rests.


In [ ]:
# ── 5.1  RCSP: enumerate all legal single-day duties ────────────────────────
#
# A single-day duty starting at DUS:
#   check-in at DUS, fly 1-6 legs (possibly via ground transfer), check-out
#   FDT ≤ table limit, and all connections respect MCT.
#
# We use DFS on the time-space network, tracking resources:
#   - current airport
#   - elapsed FDT (from check-in to current arrival + checkout)
#   - elapsed flight time
#   - number of sectors flown
#   - legs covered
#   - total cost

def generate_duties_rcsp(flights_df, ground_map):
    """
    Enumerate all EASA-legal single-day duties that start at HOME_BASE.
    Returns a list of duty dicts.
    """
    flights = flights_df.to_dict('records')
    # Index by departure airport for fast lookup
    by_dep_ap = defaultdict(list)
    for f in flights:
        by_dep_ap[f['DEPARTURE_AIRPORT']].append(f)

    duties = []

    def dfs(path, cur_airport, cur_dep_min,
            fdt_start, flight_time, sectors, cost, legs):
        """
        path       : list of leg dicts flown so far in this duty
        cur_airport: where the captain currently is
        cur_dep_min: absolute minute of the most recent departure (not arrival!)
                     — we track departure of the NEXT leg for connection checks
        fdt_start  : absolute minute of duty activation (check-in)
        flight_time: cumulative flight minutes so far
        sectors    : number of legs flown
        cost       : cumulative duty cost so far
        legs       : list of LEG_IDs flown
        """

        # ── At any point, this is a valid duty endpoint ──
        # Check-out = 30 min after last arrival
        if path:
            last = path[-1]
            checkout = last['ARR_MIN'] + CHECKOUT
            fdt = checkout - fdt_start
            limit = max_fdt_minutes(fdt_start, sectors)

            if fdt <= limit:
                # Compute cost components
                idle_time = (checkout - fdt_start) - flight_time
                hotel = COST_HOTEL_PER_NIGHT if last['ARRIVAL_AIRPORT'] != HOME_BASE else 0.0
                total_cost = (flight_time * COST_PER_FLIGHT_MIN
                              + max(0, idle_time) * COST_IDLE_PER_MIN
                              + hotel
                              + cost)   # cost already includes ground transfer costs

                duties.append({
                    'legs'         : list(legs),
                    'n_sectors'    : sectors,
                    'dep_airport'  : path[0]['DEPARTURE_AIRPORT'],
                    'arr_airport'  : last['ARRIVAL_AIRPORT'],
                    'checkin_min'  : fdt_start,
                    'checkout_min' : checkout,
                    'fdt_min'      : fdt,
                    'flight_min'   : flight_time,
                    'cost'         : round(total_cost, 2),
                    'is_roundtrip' : last['ARRIVAL_AIRPORT'] == HOME_BASE,
                    'itinerary'    : ' → '.join(
                        [path[0]['DEPARTURE_AIRPORT']]
                        + [f['ARRIVAL_AIRPORT'] for f in path])
                })

        # ── Try to extend with another leg ──
        if sectors >= 6:          # EASA practical max intra-day sectors
            return

        if path:
            last = path[-1]
            arr_ap   = last['ARRIVAL_AIRPORT']
            arr_min  = last['ARR_MIN']
        else:
            arr_ap   = HOME_BASE
            arr_min  = None      # will be set by first leg's checkin logic

        # Find candidate next legs
        candidates = list(by_dep_ap.get(arr_ap, []))

        # Also candidates reachable by ground transfer
        for (from_ap, to_ap), gt_min in ground_map.items():
            if from_ap == arr_ap:
                candidates += by_dep_ap.get(to_ap, [])

        for nxt in candidates:
            if path:
                gap = nxt['DEP_MIN'] - arr_min
                if nxt['DEPARTURE_AIRPORT'] == arr_ap:
                    if gap < MCT or gap > 240:
                        continue
                    gt_cost = 0.0
                    gt_min  = 0.0
                else:
                    # Ground transfer
                    gt_min_val = ground_map.get((arr_ap, nxt['DEPARTURE_AIRPORT']), None)
                    if gt_min_val is None or gap < gt_min_val + MCT or gap > 240:
                        continue
                    gt_cost = COST_GROUND_FIXED + gt_min_val * COST_GROUND_PER_MIN
                    gt_min  = gt_min_val
            else:
                # First leg: must depart HOME_BASE
                if nxt['DEPARTURE_AIRPORT'] != HOME_BASE:
                    continue
                gt_cost = 0.0
                gt_min  = 0.0

            # Compute check-in for this duty if it's the first leg
            checkin = (nxt['DEP_MIN'] - CHECKIN_FLIGHT) if not path else fdt_start

            # Projected FDT including this next leg
            projected_checkout = nxt['ARR_MIN'] + CHECKOUT
            projected_fdt      = projected_checkout - checkin
            projected_sectors  = sectors + 1
            fdt_limit          = max_fdt_minutes(checkin, projected_sectors)

            if projected_fdt > fdt_limit:
                continue     # EASA FDT exceeded — prune this branch

            new_flight_time = flight_time + nxt['FLT_DUR']
            new_cost        = gt_cost

            dfs(
                path    + [nxt],
                nxt['ARRIVAL_AIRPORT'],
                nxt['DEP_MIN'],
                checkin,
                new_flight_time,
                projected_sectors,
                new_cost,
                legs + [nxt['LEG_ID']]
            )

    # Start DFS from HOME_BASE for each feasible first leg
    print(f"Running RCSP duty enumeration from {HOME_BASE}...")
    dfs([], HOME_BASE, None, None, 0.0, 0, 0.0, [])

    print(f"  Raw duties generated: {len(duties)}")
    return duties

duties_raw = generate_duties_rcsp(leg_pool, GROUND_MAP)
duties_df  = pd.DataFrame(duties_raw)
print(duties_df[['itinerary','n_sectors','fdt_min','cost','is_roundtrip']].head(10).to_string())


In [ ]:
# ── 5.2  Build multi-day pairings from single-day duties ────────────────────
#
# A pairing = sequence of duties separated by overnight rests (≥ 12 h)
# that starts and ends at HOME_BASE.
# We extend duties greedily day-by-day up to MAX_PAIRING_DAYS.
#
# WHY NOT JUST USE SINGLE DUTIES?
# A single day might not have a return flight to DUS, so the captain would be
# stranded at an outstation. Multi-day pairings chain an outbound duty
# (DUS → outstation) with a return duty (outstation → DUS) the next day.

def build_pairings(duties_df, max_days=4):
    """
    Build multi-day pairings by chaining single-day duties.
    Each pairing must start and end at HOME_BASE.
    """
    duties = duties_df.to_dict('records')
    # Round-trip duties are already complete 1-day pairings
    pairings = []

    # Index duties by (dep_airport, start_day)
    by_dep_day = defaultdict(list)
    for d in duties:
        day = int(d['checkin_min'] // 1440) + 1
        d['day'] = day
        by_dep_day[(d['dep_airport'], day)].append(d)

    def recurse(chain, cur_duty, depth):
        """chain = list of duty dicts forming the current pairing"""
        cur_arr = cur_duty['arr_airport']

        # Valid endpoint: returned to HOME_BASE
        if cur_arr == HOME_BASE:
            covered = []
            total_cost = 0.0
            days = 0
            itins = []
            for d in chain:
                covered += d['legs']
                total_cost += d['cost']
                days = max(days, d['day'])
                itins.append(d['itinerary'])
            pairings.append({
                'duty_chain'   : [d for d in chain],
                'n_duties'     : len(chain),
                'total_cost'   : round(total_cost, 2),
                'covered_legs' : covered,
                'n_legs'       : len(covered),
                'home_base'    : HOME_BASE,
                'itinerary'    : ' // '.join(itins)
            })
            if depth >= max_days:
                return
        elif depth >= max_days:
            return

        # Try to extend with a duty on the NEXT day from the current arrival airport
        next_day = cur_duty['day'] + 1
        for nxt in by_dep_day.get((cur_arr, next_day), []):
            # Verify rest: next check-in must be ≥ MIN_REST after current checkout
            rest = nxt['checkin_min'] - cur_duty['checkout_min']
            if rest < MIN_REST:
                continue
            recurse(chain + [nxt], nxt, depth + 1)

    # Seed: all duties starting at HOME_BASE on Day 1
    day1_from_home = by_dep_day.get((HOME_BASE, 1), [])
    print(f"Day-1 duties departing {HOME_BASE}: {len(day1_from_home)}")

    for d in day1_from_home:
        recurse([d], d, 1)

    print(f"Total pairings generated: {len(pairings)}")
    return pd.DataFrame(pairings)

pairings_df = build_pairings(duties_df, max_days=4)
print("\nSample pairings:")
print(pairings_df[['itinerary','n_duties','n_legs','total_cost']].head(10).to_string())


In [ ]:
# ── 5.3  Remove duplicate leg coverage within a pairing ─────────────────────
# A leg should appear at most once per pairing (a captain can't fly the same
# flight twice in one pairing). Remove pairings with duplicate leg IDs.

pairings_df = pairings_df[
    pairings_df['covered_legs'].apply(lambda legs: len(legs) == len(set(legs)))
].copy().reset_index(drop=True)

# Add a safety artificial column per leg (very high cost) to guarantee feasibility
# This is the "Big-M" trick in Column Generation: if a leg cannot be covered
# organically, the solver picks the artificial column and signals the gap.
artificial_cols = []
for lid in LEG_IDS_TO_COVER:
    artificial_cols.append({
        'duty_chain'   : [],
        'n_duties'     : 0,
        'total_cost'   : PENALTY_ARTIFICIAL,
        'covered_legs' : [lid],
        'n_legs'       : 1,
        'home_base'    : HOME_BASE,
        'itinerary'    : f'ARTIFICIAL[{lid}]'
    })

print(f"Organic pairings (no duplicates): {len(pairings_df)}")
print(f"Artificial columns added        : {len(artificial_cols)}")


## 6 · Set Partitioning Problem + Column Generation

### The Set Partitioning Problem (SPP)

Let `P` be the full set of legal pairings, `L` the set of flight legs.

**Decision variables:** `x_p ∈ {0, 1}` — is pairing `p` selected?

**Objective:** minimise `Σ_p  c_p · x_p`  (total scheduling cost)

**Constraints:**  
`Σ_{p: l ∈ p}  x_p = 1   ∀ l ∈ L`   ← each leg covered *exactly once*  
`x_p ∈ {0, 1}             ∀ p ∈ P`

Because `|P|` is exponentially large, we use **Column Generation**:

1. Start with a small *Restricted Master Problem* (RMP) using the artificials + a few organic columns.
2. Solve the **LP relaxation** of the RMP (allow `x_p ∈ [0, 1]`).
3. Extract **dual variables** `π_l` for each covering constraint.
4. Compute the **reduced cost** of any new pairing `p`:  
   `r̄_p = c_p − Σ_{l ∈ p} π_l`
5. Solve the **Pricing Sub-Problem**: find the pairing with the most negative reduced cost.
6. If `r̄_p < 0`, add it to the RMP and repeat. Otherwise, LP optimum is reached.
7. Solve the **integer** RMP (MILP) to get the final schedule.


In [ ]:
# ── 6.1  Build RMP data structures ──────────────────────────────────────────
import ast

# Convert to a list of column dicts for easy manipulation
rmp_cols = pairings_df.to_dict('records') + artificial_cols

# Build a leg → column index map (for fast constraint construction)
def build_leg_col_map(cols, leg_ids):
    mapping = defaultdict(list)
    for idx, col in enumerate(cols):
        for lid in col['covered_legs']:
            if lid in leg_ids:
                mapping[lid].append(idx)
    return mapping

LEG_IDS_COVER_SET = set(LEG_IDS_TO_COVER)
leg_col_map = build_leg_col_map(rmp_cols, LEG_IDS_COVER_SET)

print(f"RMP initialised: {len(rmp_cols)} columns, {len(LEG_IDS_TO_COVER)} constraints")
print(f"Legs with 0 covering columns: "
      f"{sum(1 for l in LEG_IDS_TO_COVER if not leg_col_map[l])}")


In [ ]:
# ── 6.2  LP Solver wrappers ─────────────────────────────────────────────────
#
# We wrap both Gurobi and PuLP so the notebook works with either.

def solve_rmp_lp(cols, leg_ids, solver=SOLVER):
    """
    Solve the LP relaxation of the RMP.
    Returns: (obj_val, x_values dict col_idx→val, duals dict leg_id→pi)
    """
    n_cols = len(cols)
    lcm    = build_leg_col_map(cols, leg_ids)

    if solver == 'gurobi':
        env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
        m = gp.Model('RMP_LP', env=env)
        m.setParam('Method', 1)   # Dual simplex → gives dual variables

        x = m.addVars(n_cols, lb=0.0, ub=1.0,
                      obj=[cols[i]['total_cost'] for i in range(n_cols)],
                      vtype=GRB.CONTINUOUS)
        m.modelSense = GRB.MINIMIZE

        constrs = {}
        for lid in leg_ids:
            if lcm[lid]:
                expr = gp.quicksum(x[i] for i in lcm[lid])
                constrs[lid] = m.addConstr(expr == 1.0, name=f'cov_{lid}')

        m.optimize()
        if m.status != GRB.OPTIMAL:
            print(f"⚠️  RMP LP status: {m.status}")
            return None, None, None

        x_vals = {i: x[i].X for i in range(n_cols)}
        duals  = {lid: constrs[lid].Pi for lid in leg_ids if lid in constrs}
        return m.objVal, x_vals, duals

    else:  # PuLP
        prob = pulp.LpProblem('RMP_LP', pulp.LpMinimize)
        x = [pulp.LpVariable(f'x{i}', 0, 1) for i in range(n_cols)]
        prob += pulp.lpSum(cols[i]['total_cost'] * x[i] for i in range(n_cols))

        constrs = {}
        for lid in leg_ids:
            if lcm[lid]:
                constrs[lid] = (pulp.lpSum(x[i] for i in lcm[lid]) == 1)
                prob += constrs[lid], f'cov_{lid}'

        prob.solve(pulp.PULP_CBC_CMD(msg=0))
        if pulp.LpStatus[prob.status] != 'Optimal':
            print(f"⚠️  RMP LP: {pulp.LpStatus[prob.status]}")
            return None, None, None

        x_vals = {i: pulp.value(x[i]) for i in range(n_cols)}
        duals  = {lid: constrs[lid].pi if hasattr(constrs[lid], 'pi')
                  else constrs[lid].constant   # PuLP constraint dual
                  for lid in constrs}
        return pulp.value(prob.objective), x_vals, duals

print("LP solver wrapper ready.")


In [ ]:
# ── 6.3  Pricing Sub-Problem (RCSP on reduced costs) ────────────────────────
#
# Given dual variables π_l for each leg l, the reduced cost of pairing p is:
#   r̄_p = c_p − Σ_{l ∈ p} π_l
#
# We want to find pairings with r̄_p < 0 (they improve the LP objective).
#
# APPROACH: Re-run the DFS duty enumeration but replace the cost of each
# duty with its REDUCED COST = duty_cost − sum of duals on its legs.
# Then chain duties into pairings and return any pairing with negative total
# reduced cost. This is the Pricing Sub-Problem (PSP), which is itself an RCSP.

def solve_pricing_subproblem(duties_df, duals, ground_map, max_days=4,
                             max_cols_per_iter=20, threshold=-150.0):
    """
    Find new pairings with negative reduced cost.
    duals: dict {leg_id: π_l}
    Returns list of new pairing dicts to add to the RMP.
    """
    duties = duties_df.to_dict('records')

    # Compute reduced cost for each duty
    for d in duties:
        dual_gain = sum(duals.get(lid, 0.0) for lid in d['legs']
                        if lid in LEG_IDS_COVER_SET)
        d['rc'] = d['cost'] - dual_gain

    # Index by (dep_airport, day)
    by_dep_day = defaultdict(list)
    for d in duties:
        day = int(d['checkin_min'] // 1440) + 1
        d['day'] = day
        by_dep_day[(d['dep_airport'], day)].append(d)

    new_cols = []

    def dfs_price(chain, cur_duty, cum_rc, cum_cost, cum_legs, depth):
        if len(new_cols) >= max_cols_per_iter:
            return

        cur_arr = cur_duty['arr_airport']

        # Valid endpoint: back at HOME_BASE with negative reduced cost
        if cur_arr == HOME_BASE and cum_rc < threshold:
            new_cols.append({
                'duty_chain'   : list(chain),
                'n_duties'     : len(chain),
                'total_cost'   : round(cum_cost, 2),
                'covered_legs' : list(cum_legs),
                'n_legs'       : len(cum_legs),
                'home_base'    : HOME_BASE,
                'itinerary'    : ' // '.join(d['itinerary'] for d in chain)
            })
            if depth >= max_days:
                return
        elif depth >= max_days:
            return

        # Prune: even if reduced cost is already very positive, don't explore further
        if cum_rc > 5000:
            return

        next_day = cur_duty['day'] + 1
        for nxt in by_dep_day.get((cur_arr, next_day), []):
            rest = nxt['checkin_min'] - cur_duty['checkout_min']
            if rest < MIN_REST:
                continue
            dfs_price(chain + [nxt], nxt,
                      cum_rc   + nxt['rc'],
                      cum_cost + nxt['cost'],
                      cum_legs + nxt['legs'],
                      depth + 1)

    # Start from Day 1 duties departing HOME_BASE
    for start_duty in by_dep_day.get((HOME_BASE, 1), []):
        if start_duty['rc'] > 3000:
            continue   # Fast prune: too expensive even without extension
        dfs_price([start_duty], start_duty, start_duty['rc'],
                  start_duty['cost'], list(start_duty['legs']), 1)
        if len(new_cols) >= max_cols_per_iter:
            break

    return new_cols

print("Pricing sub-problem function ready.")


In [ ]:
# ── 6.4  Column Generation Loop ──────────────────────────────────────────────

print("=" * 60)
print("COLUMN GENERATION LOOP")
print("=" * 60)

MAX_CG_ITERATIONS = 20
cg_history = []   # Track objective value per iteration

for iteration in range(1, MAX_CG_ITERATIONS + 1):
    # Step A: Solve LP relaxation of current RMP
    obj, x_vals, duals = solve_rmp_lp(rmp_cols, LEG_IDS_TO_COVER)

    if obj is None:
        print(f"Iter {iteration:02d}: LP infeasible / error — stopping.")
        break

    cg_history.append(obj)
    pct_artificial = sum(
        x_vals[i] for i, c in enumerate(rmp_cols)
        if 'ARTIFICIAL' in c['itinerary']
    )

    # Step B: Pricing sub-problem — find negative reduced-cost columns
    new_cols = solve_pricing_subproblem(duties_df, duals, GROUND_MAP,
                                        max_days=4, max_cols_per_iter=30,
                                        threshold=-100.0)

    print(f"Iter {iteration:02d} | LP obj = {obj:>12,.0f} | "
          f"Artificial use = {pct_artificial:.3f} | "
          f"New columns = {len(new_cols)}")

    if not new_cols:
        print(f"\n✅ No improving columns found — LP optimum reached after {iteration} iterations.")
        break

    # Step C: Add new columns to the RMP
    rmp_cols.extend(new_cols)
    leg_col_map = build_leg_col_map(rmp_cols, LEG_IDS_COVER_SET)

print(f"\nFinal RMP size: {len(rmp_cols)} columns")


## 7 · Integer Solve (MILP / Branch-and-Bound)

In [ ]:
# ── 7.1  Solve the integer RMP ──────────────────────────────────────────────
#
# Now that CG has produced a good column pool, we fix integrality (x ∈ {0,1}).
# This is a Set Partitioning MILP. Gurobi/CBC solves it via branch-and-bound.
#
# We retain the artificial columns (with their large penalty costs) so that
# the MILP is always feasible; any artificial column chosen in the solution
# signals a leg that cannot be covered by the current pairing pool.

def solve_milp(cols, leg_ids, solver=SOLVER):
    """
    Solve the integer Set Partitioning Problem.
    Returns (obj_val, chosen_column_indices)
    """
    n_cols = len(cols)
    lcm    = build_leg_col_map(cols, leg_ids)

    if solver == 'gurobi':
        env = gp.Env(empty=True); env.setParam('OutputFlag', 0); env.start()
        m = gp.Model('CSP_MILP', env=env)
        m.setParam('MIPGap', 0.001)
        m.setParam('Presolve', 2)

        x = m.addVars(n_cols, vtype=GRB.BINARY,
                      obj=[cols[i]['total_cost'] for i in range(n_cols)])
        m.modelSense = GRB.MINIMIZE

        for lid in leg_ids:
            if lcm[lid]:
                m.addConstr(gp.quicksum(x[i] for i in lcm[lid]) == 1.0)

        m.optimize()
        chosen = [i for i in range(n_cols) if x[i].X > 0.5]
        return m.objVal, chosen

    else:  # PuLP
        prob = pulp.LpProblem('CSP_MILP', pulp.LpMinimize)
        x = [pulp.LpVariable(f'x{i}', cat='Binary') for i in range(n_cols)]
        prob += pulp.lpSum(cols[i]['total_cost'] * x[i] for i in range(n_cols))
        for lid in leg_ids:
            if lcm[lid]:
                prob += pulp.lpSum(x[i] for i in lcm[lid]) == 1.0
        prob.solve(pulp.PULP_CBC_CMD(msg=0))
        chosen = [i for i in range(n_cols) if pulp.value(x[i]) > 0.5]
        return pulp.value(prob.objective), chosen

print("Solving integer MILP...")
milp_obj, chosen_idx = solve_milp(rmp_cols, LEG_IDS_TO_COVER)
chosen_pairings = [rmp_cols[i] for i in chosen_idx]

print(f"\n🎉 MILP Objective (total cost): {milp_obj:,.0f}")
print(f"Total pairings selected: {len(chosen_pairings)}")


## 8 · Solution Analysis & Validation

In [ ]:
# ── 8.1  Coverage check ─────────────────────────────────────────────────────

covered_legs = []
for p in chosen_pairings:
    covered_legs.extend(p['covered_legs'])

covered_set   = set(l for l in covered_legs if l in LEG_IDS_COVER_SET)
covered_count = len(covered_set)
total_count   = len(LEG_IDS_TO_COVER)

print("=" * 50)
print("SOLUTION VALIDATION")
print("=" * 50)
print(f"Legs to cover : {total_count}")
print(f"Legs covered  : {covered_count}")
print(f"Coverage      : {100*covered_count/total_count:.1f}%")

# Check for double-covered legs (should be 0 in a valid SPP solution)
from collections import Counter
leg_counts = Counter(l for l in covered_legs if l in LEG_IDS_COVER_SET)
double_covered = {l: c for l, c in leg_counts.items() if c > 1}
uncovered      = LEG_IDS_TO_COVER - covered_set

print(f"Double-covered legs : {len(double_covered)}")
print(f"Uncovered legs      : {len(uncovered)}")
if uncovered:
    print("⚠️  Uncovered legs (check your duty generation):", list(uncovered)[:5])


In [ ]:
# ── 8.2  Pairing summary ────────────────────────────────────────────────────

organic    = [p for p in chosen_pairings if 'ARTIFICIAL' not in p['itinerary']]
artificial = [p for p in chosen_pairings if 'ARTIFICIAL'     in p['itinerary']]

print(f"Organic pairings selected   : {len(organic)}")
print(f"Artificial columns selected : {len(artificial)}  ← should be 0 in a perfect solve")

if organic:
    pairings_summary = pd.DataFrame([{
        'itinerary'  : p['itinerary'],
        'n_duties'   : p.get('n_duties', 1),
        'n_legs'     : p['n_legs'],
        'cost'       : p['total_cost']
    } for p in organic])

    print("\nTop 20 selected pairings by cost:")
    print(pairings_summary.sort_values('cost', ascending=False).head(20).to_string(index=False))

    print(f"\nCost breakdown:")
    print(f"  Total organic cost   : {pairings_summary['cost'].sum():>10,.0f}")
    print(f"  Avg cost per pairing : {pairings_summary['cost'].mean():>10,.1f}")
    print(f"  Max cost pairing     : {pairings_summary['cost'].max():>10,.1f}")
    print(f"  Avg legs per pairing : {pairings_summary['n_legs'].mean():>10.2f}")


In [ ]:
# ── 8.3  Column Generation convergence plot ──────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

if cg_history:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(range(1, len(cg_history)+1), cg_history, 'o-', color='#2c7bb6', lw=2)
    ax.axhline(milp_obj, color='#d7191c', ls='--', label=f'MILP integer obj = {milp_obj:,.0f}')
    ax.set_xlabel('CG Iteration')
    ax.set_ylabel('LP Relaxation Objective')
    ax.set_title('Column Generation Convergence (DUS · Week 1)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('cg_convergence.png', dpi=150)
    plt.show()
    print("Plot saved to cg_convergence.png")


In [ ]:
# ── 8.4  Duty-level EASA compliance check ───────────────────────────────────

print("\nEASA compliance check on selected duties:")
violations = []

for p in organic:
    for d in p.get('duty_chain', []):
        if not isinstance(d, dict):
            continue
        fdt   = d.get('fdt_min', 0)
        secs  = d.get('n_sectors', 1)
        cin   = d.get('checkin_min', 0)
        limit = max_fdt_minutes(cin, secs)
        if fdt > limit:
            violations.append({
                'itinerary': d.get('itinerary', '?'),
                'fdt_min'  : fdt,
                'limit_min': limit,
                'excess_min': fdt - limit
            })

if violations:
    print(f"⚠️  {len(violations)} FDT violations found!")
    print(pd.DataFrame(violations).head(10).to_string())
else:
    print("✅ All selected duties comply with EASA FDT limits.")


In [ ]:
# ── 8.5  Save results ────────────────────────────────────────────────────────
import json as _json

output_rows = []
for p in organic:
    output_rows.append({
        'home_base'    : p['home_base'],
        'n_duties'     : p.get('n_duties', 1),
        'n_legs'       : p['n_legs'],
        'total_cost'   : p['total_cost'],
        'covered_legs' : str(p['covered_legs']),
        'itinerary'    : p['itinerary']
    })

results_df = pd.DataFrame(output_rows)
results_df.to_csv('optimal_pairings_DUS_week1.csv', index=False, encoding='utf-8-sig')
print(f"Results saved: optimal_pairings_DUS_week1.csv  ({len(results_df)} rows)")
print("\n✅ Done! Scroll up to review coverage and cost summaries.")


## 9 · What comes next

### Scaling to all bases
Change `HOME_BASE` to `'STR'`, `'HAM'`, `'CGN'`, `'BER'` and re-run.  
Each base is largely independent — pairings must start **and end** at the same base —  
so you can run them in parallel and merge the results.

### Scaling to the full 3-month horizon
The column generation loop scales well: only the number of *legs to cover* grows  
(more constraints), not the number of columns per iteration (the pricing sub-problem  
stays the same RCSP). Expect longer CG iterations but the same algorithm structure.

### Rostering (Phase 2)
Once you have the optimal pairing set `P*`, the **rostering problem** is:

| Element | Description |
|---|---|
| Variable | `y[c, p] ∈ {0,1}` — assign captain `c` to pairing `p` |
| Constraint | Each pairing assigned to exactly 1 captain: `Σ_c y[c,p] = 1  ∀ p` |
| Constraint | Each captain assigned to at most 1 pairing per 7-day block |
| Constraint | Captain home base = pairing home base |
| Hard constraint | No duty on approved vacation days (from `codes.csv` KUR/U) |
| Soft penalty | Minimize deviation from `off_claims` target |
| Soft penalty | Maximize granted `off_requests` |

The rostering problem is much smaller than pairing (few hundred captains × few thousand pairings) and can typically be solved directly as a MILP without column generation.

### Integrated approach (for bonus marks)
Instead of sequential pairing→rostering, assign captain labels to columns during the  
pricing sub-problem itself. This makes RCSP captain-specific (resources now include  
individual duty histories and rest carryover) but produces Pareto-better schedules.
